In [1]:
import pandas as pd
import os
from Bio import SeqIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
import numpy as np
from tqdm import tqdm

def calculate_gc_content(sequence):
    return (sequence.count("G") + sequence.count("C")) / len(sequence) * 100 if sequence else 0

def match_sequences_by_gc(mpra_file, fasta_file, output_fasta):
    # Load sequence data with GC content
    df = pd.read_csv(mpra_file, index_col=0)
    df["GC_Content"] = df["Sequence_Major_padded"].apply(calculate_gc_content)

    # Load and compute GC content for FASTA sequences
    fasta_sequences = [
        {
            "ID": record.id,
            "Sequence": str(record.seq),
            "GC_Content": calculate_gc_content(str(record.seq))
        }
        for record in SeqIO.parse(fasta_file, "fasta")
    ]

    # Sort by GC content and reset index
    fasta_df = pd.DataFrame(fasta_sequences).sort_values(by="GC_Content").reset_index(drop=True)
    gc_values = fasta_df["GC_Content"].values  # Sorted GC content

    used_ids = set()
    matched_sequences = []

    for _, row in tqdm(df.iterrows(), total=len(df), desc="Matching sequences"):
        target_gc = row["GC_Content"]
        target_base_250_major = row["Sequence_Major_padded"][250]
        target_base_250_minor = row["Sequence_Minor_padded"][250]

        found_match = None

        # 1) Try to find a match via the three closest GC indices from binary search
        idx = np.searchsorted(gc_values, target_gc)
        # Use unique indices in case idx-1, idx, idx+1 have duplicates
        candidate_indices = np.unique(np.clip([idx - 1, idx, idx + 1], 0, len(fasta_df) - 1))
        candidates = fasta_df.iloc[candidate_indices]
        # Filter candidates by unused IDs and matching 250th base
        candidates = candidates[
            (~candidates["ID"].isin(used_ids)) &
            (candidates["Sequence"].str[250] == target_base_250_major)
        ]
        if not candidates.empty:
            # Randomly pick one candidate from the available matches
            found_match = candidates.sample(n=1).iloc[0]
            used_ids.add(found_match["ID"])

        # 2) If still not found, expand GC range in 0.5% increments up to ±5%
        range_step = 0.5
        max_range = 5.0
        while found_match is None and range_step <= max_range:
            lower_bound = target_gc - range_step
            upper_bound = target_gc + range_step

            # Filter by relaxed GC range and unused IDs
            subset = fasta_df[
                (fasta_df["GC_Content"] >= lower_bound) &
                (fasta_df["GC_Content"] <= upper_bound) &
                (~fasta_df["ID"].isin(used_ids))
            ]

            # Check for matching base at position 250
            matching_rows = subset[subset["Sequence"].str[250] == target_base_250_major]
            if not matching_rows.empty:
                # Randomly pick one candidate from the matching rows
                found_match = matching_rows.sample(n=1).iloc[0]
                used_ids.add(found_match["ID"])
            else:
                range_step += 0.5

        # 3) If a match is found, record the details
        if found_match is not None:
            matched_sequences.append({
                "Original_Sequence": row["Sequence_Major_padded"],
                "Original_GC": target_gc,
                "Matched_Sequence_OriginalAllele": found_match["Sequence"],
                "Matched_Sequence_SyntheticMinorAllele": (
                    found_match["Sequence"][:250]
                    + target_base_250_minor
                    + found_match["Sequence"][251:]
                ),
                "Matched_GC": found_match["GC_Content"],
                "Matched_ID": found_match["ID"]
            })

    # Convert results to DataFrame
    matched_df = pd.DataFrame(matched_sequences)

    # Report how many matches were found
    total_sequences = len(df)
    matched_count = len(matched_df)
    print(f"Number of matched sequences: {matched_count} out of {total_sequences}")

    # Save matched sequences to FASTA if any matches found
    if matched_count > 0:
        fasta_records = []
        for _, row in matched_df.iterrows():
            for col in ["Matched_Sequence_OriginalAllele"]:
                seq_str = row[col]
                record_id = f"{row['Matched_ID']}_{col.split('_')[-1]}_{seq_str[250]}_GC{int(row['Matched_GC'])}"
                fasta_records.append(SeqRecord(Seq(seq_str), id=record_id, description=""))
        for _, row in matched_df.iterrows():
            for col in ["Matched_Sequence_SyntheticMinorAllele"]:
                seq_str = row[col]
                record_id = f"{row['Matched_ID']}_{col.split('_')[-1]}_{seq_str[250]}_GC{int(row['Matched_GC'])}"
                fasta_records.append(SeqRecord(Seq(seq_str), id=record_id, description=""))
        SeqIO.write(fasta_records, output_fasta, "fasta")
        print(f"FASTA file saved as {output_fasta}")
    else:
        print("No sequences were matched. No FASTA file generated.")

def process_all_fasta(end_keywords, mpra_file, fasta_dir, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    for fasta_file in os.listdir(fasta_dir):
        if fasta_file.endswith(end_keywords):
            fasta_path = os.path.join(fasta_dir, fasta_file)
            output_name = fasta_file.split(".")[0] + "_MPRA_Major_Minor_500bp_SNPCenter_NegativeATACBackground_NoOverlap_GCmatched.fasta"
            output_path = os.path.join(output_dir, output_name)
            match_sequences_by_gc(mpra_file, fasta_path, output_path)


In [2]:
# Example usage

# Example usage
process_all_fasta(
    end_keywords = "GCmatch_background_noOverlap_validation.fasta",
    mpra_file="/media/zihengc/T7/mpra3_lib_analysis/indexing/MPRA_SNPCenter_500bpSequences_withManualAnnotation_500bp.csv",
    fasta_dir="/media/zihengc/T7/THP1_machinelearning/background_negatives_500bp_mm10",
    output_dir = '/media/zihengc/T7/THP1_machinelearning/mpra3_prediction/sequence_for_prediction/MPRA_Major_Minor_500bp_SNPCenter_NegativeATACBackground_NoOverlap_GCmatched_855Sequences'
)
# Example usage
process_all_fasta(
    end_keywords = "neg.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap_train.fasta",
    mpra_file="/media/zihengc/T7/mpra3_lib_analysis/indexing/MPRA_SNPCenter_500bpSequences_withManualAnnotation_500bp.csv",
    fasta_dir="/media/zihengc/T7/THP1_machinelearning/background_negatives_500bp_hg38",
    output_dir = '/media/zihengc/T7/THP1_machinelearning/mpra3_prediction/sequence_for_prediction/MPRA_Major_Minor_500bp_SNPCenter_NegativeATACBackground_NoOverlap_GCmatched_855Sequences'
)
# Example usage
process_all_fasta(
    end_keywords = "pos.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap_train.fasta",
    mpra_file="/media/zihengc/T7/mpra3_lib_analysis/indexing/MPRA_SNPCenter_500bpSequences_withManualAnnotation_500bp.csv",
    fasta_dir="/media/zihengc/T7/THP1_machinelearning/background_negatives_500bp_hg38",
    output_dir = '/media/zihengc/T7/THP1_machinelearning/mpra3_prediction/sequence_for_prediction/MPRA_Major_Minor_500bp_SNPCenter_NegativeATACBackground_NoOverlap_GCmatched_855Sequences')

Matching sequences: 100%|██████████| 855/855 [00:01<00:00, 775.21it/s]


Number of matched sequences: 855 out of 855
FASTA file saved as /media/zihengc/T7/THP1_machinelearning/mpra3_prediction/sequence_for_prediction/MPRA_Major_Minor_500bp_SNPCenter_NegativeATACBackground_NoOverlap_GCmatched_855Sequences/Pfenning_bulk_CpuCombined_Striatum_MPRA_Major_Minor_500bp_SNPCenter_NegativeATACBackground_NoOverlap_GCmatched.fasta


Matching sequences: 100%|██████████| 855/855 [00:01<00:00, 748.69it/s]


Number of matched sequences: 855 out of 855
FASTA file saved as /media/zihengc/T7/THP1_machinelearning/mpra3_prediction/sequence_for_prediction/MPRA_Major_Minor_500bp_SNPCenter_NegativeATACBackground_NoOverlap_GCmatched_855Sequences/Pfenning_bulk_CtxCombined_Cortex_MPRA_Major_Minor_500bp_SNPCenter_NegativeATACBackground_NoOverlap_GCmatched.fasta


Matching sequences: 100%|██████████| 855/855 [00:01<00:00, 768.02it/s]


...

Number of matched sequences: 855 out of 855
FASTA file saved as /media/zihengc/T7/THP1_machinelearning/mpra3_prediction/sequence_for_prediction/MPRA_Major_Minor_500bp_SNPCenter_NegativeATACBackground_NoOverlap_GCmatched_855Sequences/THP1_LPSIFNG_vs_IFNG_neg_MPRA_Major_Minor_500bp_SNPCenter_NegativeATACBackground_NoOverlap_GCmatched.fasta


Matching sequences: 100%|██████████| 855/855 [00:01<00:00, 821.32it/s] 


Number of matched sequences: 855 out of 855
FASTA file saved as /media/zihengc/T7/THP1_machinelearning/mpra3_prediction/sequence_for_prediction/MPRA_Major_Minor_500bp_SNPCenter_NegativeATACBackground_NoOverlap_GCmatched_855Sequences/THP1_LPSIFNG_vs_Naive_pos_MPRA_Major_Minor_500bp_SNPCenter_NegativeATACBackground_NoOverlap_GCmatched.fasta


Matching sequences: 100%|██████████| 855/855 [00:01<00:00, 794.25it/s]

Number of matched sequences: 855 out of 855
FASTA file saved as /media/zihengc/T7/THP1_machinelearning/mpra3_prediction/sequence_for_prediction/MPRA_Major_Minor_500bp_SNPCenter_NegativeATACBackground_NoOverlap_GCmatched_855Sequences/THP1_LPSIFNG_vs_IFNG_pos_MPRA_Major_Minor_500bp_SNPCenter_NegativeATACBackground_NoOverlap_GCmatched.fasta


In [ ]:
process_all_fasta(
    end_keywords = "GCmatch_background_noOverlap_validation.fasta",
    mpra_file="/media/zihengc/T7/mpra3_lib_analysis/indexing/MPRA_SNPCenter_500bpSequences_withManualAnnotation_500bp.csv",
    fasta_dir="/media/zihengc/T7/THP1_machinelearning/background_negatives_500bp_hg38",
    output_dir = '/media/zihengc/T7/THP1_machinelearning/mpra3_prediction/sequence_for_prediction/MPRA_Major_Minor_500bp_SNPCenter_NegativeATACBackground_NoOverlap_GCmatched_855Sequences'
)

In [ ]:
process_all_fasta(#revision
    end_keywords="GCmatch_background_noOverlap_validation.fasta",
    mpra_file="/media/zihengc/T7/mpra3_lib_analysis/indexing/MPRA_SNPCenter_500bpSequences_withManualAnnotation_500bp.csv",
    fasta_dir="/media/zihengc/T7/THP1_machinelearning/background_negatives_500bp_hg38",
    output_dir="./"
)

Matching sequences: 100%|██████████| 855/855 [00:01<00:00, 740.23it/s]


Number of matched sequences: 855 out of 855
FASTA file saved as ./THP1_IFNG_4hrs_MPRA_Major_Minor_500bp_SNPCenter_NegativeATACBackground_NoOverlap_GCmatched.fasta


Matching sequences: 100%|██████████| 855/855 [00:00<00:00, 872.08it/s] 


Number of matched sequences: 855 out of 855
FASTA file saved as ./THP1_LPSIFNG_4hrs_MPRA_Major_Minor_500bp_SNPCenter_NegativeATACBackground_NoOverlap_GCmatched.fasta


Matching sequences: 100%|██████████| 855/855 [00:00<00:00, 993.55it/s] 


...

Number of matched sequences: 855 out of 855
FASTA file saved as ./fullard_hippocampus_neurons_MPRA_Major_Minor_500bp_SNPCenter_NegativeATACBackground_NoOverlap_GCmatched.fasta


Matching sequences: 100%|██████████| 855/855 [00:00<00:00, 856.05it/s] 


Number of matched sequences: 855 out of 855
FASTA file saved as ./astrocyte_GSE188398_MPRA_Major_Minor_500bp_SNPCenter_NegativeATACBackground_NoOverlap_GCmatched.fasta


Matching sequences: 100%|██████████| 855/855 [00:00<00:00, 978.03it/s] 

Number of matched sequences: 855 out of 855
FASTA file saved as ./oligodendrocyte_GSE143666_MPRA_Major_Minor_500bp_SNPCenter_NegativeATACBackground_NoOverlap_GCmatched.fasta
